# 08 — Earth matching for the Mars dust-compatible environment

This notebook is the reusable Earth matcher. It does **not** claim active wind transport. It ranks Earth grid
cells whose environmental conditions are compatible with the 14 Mars regions that passed the notebook 07
thermophysical dust check.

Numeric Earth data come from NASA POWER's documented climatology API. NASA CMR is then used to inventory
MODIS follow-up coverage near the highest-ranked cells. CMR metadata alone is never used as a measurement.

The Mars-to-Earth mappings are explicit proxies:

- Mars albedo → Earth surface albedo.
- Mars low thermal inertia → large Earth surface-temperature range (weak proxy; not the same physical quantity).
- Mars dust index → Earth dry + windy conditions.

Because one mapping is only a proxy, the final output is a **preliminary environmental analog ranking**.


In [ ]:
from __future__ import annotations

import json
import math
import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100)

OUTPUT_ROOT = Path("/kaggle/working/earth_dust_matching") if Path("/kaggle/working").exists() else Path.cwd() / "build" / "earth_dust_matching"
FIGURE_DIR = OUTPUT_ROOT / "figures"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

POWER_URL = "https://power.larc.nasa.gov/api/temporal/climatology/point"
POWER_PARAMETERS = "TS,PRECTOTCORR,WS10M,ALLSKY_SRF_ALB"
POWER_START = 2001
POWER_END = 2025
CMR_URL = "https://cmr.earthdata.nasa.gov/search/granules.json"
MODIS_ALBEDO_COLLECTION = "C1748066515-LPCLOUD"  # MOD13Q1 coverage inventory

print(f"Outputs: {OUTPUT_ROOT}")


In [ ]:
def find_input(filename: str, parent_hint: str) -> Path:
    roots = [Path("/kaggle/input"), Path("/kaggle/working"), Path.cwd(), Path("/tmp")]
    matches = []
    for root in roots:
        if not root.exists():
            continue
        for candidate in root.rglob(filename):
            text = candidate.as_posix()
            if parent_hint in text and OUTPUT_ROOT.as_posix() not in text:
                matches.append(candidate)
    if not matches:
        raise FileNotFoundError(f"Missing {filename}; attach notebook 07 output or set an input override")
    return sorted(set(matches), key=lambda p: ("/kaggle/input/" not in p.as_posix(), len(p.as_posix())))[0]


MARS_RESULTS_PATH = find_input("mars_candidate_validation_results.parquet", "mars_candidate_validation")
MARS_DISTRIBUTION_PATH = find_input("mars_supported_environment_distributions.parquet", "mars_candidate_validation")
mars_results_df = pd.read_parquet(MARS_RESULTS_PATH)
mars_distribution_df = pd.read_parquet(MARS_DISTRIBUTION_PATH)
mars_supported_df = mars_results_df.loc[
    mars_results_df["validated_interpretation"].eq("fine_particulate_dust_compatible_surface")
].copy()
required_mars = {
    "region_id", "centroid_latitude", "centroid_longitude", "raw_mean__albedo_mean",
    "raw_mean__thermal_inertia_mean", "raw_mean__quality_masked__mineral_surface_dust",
}
assert required_mars <= set(mars_supported_df.columns), required_mars - set(mars_supported_df.columns)
assert len(mars_supported_df) >= 5
print(f"Mars source regions: {len(mars_supported_df)}")
display(mars_distribution_df)


## Proxy contract

The matcher must not silently pretend that Earth temperature is Mars thermal inertia. We preserve the mapping
and its limitation in the output. The score is therefore useful for finding candidate environments, not for
claiming the same material or process exists on Earth.


In [ ]:
proxy_mapping_df = pd.DataFrame([
    {
        "mars_feature": "raw_mean__albedo_mean",
        "earth_feature": "earth_surface_albedo_ann",
        "direction": "same",
        "strength": "directer",
        "reason": "Both are dimensionless surface reflectance/albedo descriptors, though sensors and atmosphere differ.",
    },
    {
        "mars_feature": "raw_mean__thermal_inertia_mean",
        "earth_feature": "earth_surface_temperature_range_c",
        "direction": "inverse_mars_low_inertia_to_earth_high_range",
        "strength": "weak_proxy",
        "reason": "Thermal inertia is not temperature range; seasonal surface-temperature range is only an environmental proxy.",
    },
    {
        "mars_feature": "raw_mean__quality_masked__mineral_surface_dust",
        "earth_feature": "earth_dry_windy_proxy",
        "direction": "same",
        "strength": "process_proxy",
        "reason": "Dry and windy conditions favor exposed mobile particulate surfaces but do not prove dust cover.",
    },
])
proxy_mapping_df.to_csv(OUTPUT_ROOT / "proxy_mapping.csv", index=False)
display(proxy_mapping_df)


## 1. Obtain actual Earth measurements from NASA POWER

We query a global 15° grid. Each point is a real NASA POWER climatology response, not a fabricated value. The
API returns annual surface albedo, corrected precipitation, 10 m wind, and monthly surface temperature.


In [ ]:
grid_points = [
    {"latitude": float(latitude), "longitude": float(longitude)}
    for latitude in np.arange(-75, 76, 15)
    for longitude in np.arange(-180, 180, 15)
]


def power_request(point: dict[str, float], attempts: int = 3) -> dict[str, Any]:
    params = {
        "parameters": POWER_PARAMETERS,
        "community": "AG",
        "longitude": point["longitude"],
        "latitude": point["latitude"],
        "format": "JSON",
        "start": POWER_START,
        "end": POWER_END,
    }
    last_error = None
    for attempt in range(attempts):
        try:
            response = requests.get(POWER_URL, params=params, timeout=45)
            response.raise_for_status()
            payload = response.json()
            values = payload["properties"]["parameter"]
            geometry = payload.get("geometry", {}).get("coordinates", [np.nan, np.nan, np.nan])
            elevation = float(geometry[2]) if len(geometry) > 2 else np.nan
            return {"point": point, "values": values, "elevation_m": elevation, "status": "success", "error": ""}
        except Exception as error:
            last_error = repr(error)
            time.sleep(1.5 * (attempt + 1))
    return {"point": point, "values": {}, "status": "error", "error": last_error}


power_rows = []
with ThreadPoolExecutor(max_workers=8) as executor:
    futures = [executor.submit(power_request, point) for point in grid_points]
    for future in as_completed(futures):
        power_rows.append(future.result())
successful_power = [row for row in power_rows if row["status"] == "success"]
if len(successful_power) < 0.90 * len(grid_points):
    raise RuntimeError(f"Too many POWER failures: {len(power_rows) - len(successful_power)} / {len(power_rows)}")


def annual(parameter: dict[str, Any], name: str) -> float:
    value = parameter.get(name, {}).get("ANN", np.nan)
    return float(value) if value is not None else np.nan


earth_rows = []
for row in successful_power:
    values = row["values"]
    ts_values = [float(value) for key, value in values.get("TS", {}).items() if key != "ANN" and float(value) > -900]
    precipitation = annual(values, "PRECTOTCORR")
    wind = annual(values, "WS10M")
    albedo = annual(values, "ALLSKY_SRF_ALB")
    temperature = annual(values, "TS")
    earth_rows.append({
        "latitude": row["point"]["latitude"],
        "longitude": row["point"]["longitude"],
        "earth_surface_albedo_ann": albedo,
        "elevation_m": row.get("elevation_m", np.nan),
        "earth_precipitation_mm_day_ann": precipitation,
        "earth_wind_mps_ann": wind,
        "earth_surface_temperature_c_ann": temperature,
        "earth_surface_temperature_range_c": max(ts_values) - min(ts_values) if ts_values else np.nan,
        "power_status": row["status"],
    })
earth_df = pd.DataFrame(earth_rows)
earth_df["earth_annual_precipitation_mm"] = earth_df["earth_precipitation_mm_day_ann"] * 365.25
earth_df["earth_aridity_score"] = -np.log1p(earth_df["earth_annual_precipitation_mm"].clip(lower=0))
earth_df["earth_dry_windy_proxy"] = (
    earth_df["earth_aridity_score"].rank(pct=True)
    + earth_df["earth_wind_mps_ann"].rank(pct=True)
) / 2
earth_df = earth_df.replace([np.inf, -np.inf], np.nan).dropna(
    subset=["earth_surface_albedo_ann", "earth_surface_temperature_range_c", "earth_dry_windy_proxy", "elevation_m"]
).reset_index(drop=True)
# POWER's returned elevation is a practical coarse land/ocean guard. Cells at or below sea level are
# excluded from this global screening; later work can add a proper high-resolution land mask.
earth_df = earth_df.loc[earth_df["elevation_m"] > 0].reset_index(drop=True)
assert len(earth_df) >= 100
display(earth_df.head())
print(f"NASA POWER Earth land cells: {len(earth_df)} / {len(grid_points)}")


## 2. Distribution and joint matching

All features become within-planet percentile ranks. This avoids comparing Mars and Earth raw numbers that have
different sensors and physical contexts. A cell is close when it is close to at least several Mars source
regions in the joint feature space, while its marginal values also remain inside the Mars source envelope.


In [ ]:
def percentile_rank(values: pd.Series) -> np.ndarray:
    return values.rank(method="average", pct=True).to_numpy(dtype=float)


mars_raw = mars_supported_df[[
    "raw_mean__albedo_mean", "raw_mean__thermal_inertia_mean", "raw_mean__quality_masked__mineral_surface_dust"
]].apply(pd.to_numeric, errors="coerce").dropna().reset_index(drop=True)
earth_raw = earth_df[[
    "earth_surface_albedo_ann", "earth_surface_temperature_range_c", "earth_dry_windy_proxy"
]].apply(pd.to_numeric, errors="coerce")
mars_rank = pd.DataFrame({
    "albedo": percentile_rank(mars_raw["raw_mean__albedo_mean"]),
    "low_inertia": 1.0 - percentile_rank(mars_raw["raw_mean__thermal_inertia_mean"]),
    "dry_windy": percentile_rank(mars_raw["raw_mean__quality_masked__mineral_surface_dust"]),
}).to_numpy()
earth_rank = pd.DataFrame({
    "albedo": percentile_rank(earth_raw["earth_surface_albedo_ann"]),
    "low_inertia": percentile_rank(earth_raw["earth_surface_temperature_range_c"]),
    "dry_windy": percentile_rank(earth_raw["earth_dry_windy_proxy"]),
}).to_numpy()

mars_q05, mars_q95 = np.quantile(mars_rank, [0.05, 0.95], axis=0)
marginal_penalty = np.maximum(mars_q05 - earth_rank, 0) + np.maximum(earth_rank - mars_q95, 0)
marginal_similarity = 1.0 - marginal_penalty.mean(axis=1).clip(0, 1)
joint_distances = np.sqrt(((earth_rank[:, None, :] - mars_rank[None, :, :]) ** 2).mean(axis=2))
k = min(3, mars_rank.shape[0])
nearest_joint_distance = np.partition(joint_distances, kth=k - 1, axis=1)[:, :k].mean(axis=1)
joint_similarity = 1.0 - np.clip(nearest_joint_distance / math.sqrt(1 / 3), 0, 1)

preliminary_score = (marginal_similarity + joint_similarity) / 2
earth_df["marginal_similarity"] = marginal_similarity
earth_df["joint_similarity"] = joint_similarity
earth_df["preliminary_analog_score"] = preliminary_score


In [ ]:
# A simple spatial coherence check: nearby grid cells should also be promising.
coordinate_array = earth_df[["latitude", "longitude"]].to_numpy(dtype=float)
score_array = earth_df["preliminary_analog_score"].to_numpy(dtype=float)
top_threshold = np.quantile(score_array, 0.75)
coherence = []
for index, (latitude, longitude) in enumerate(coordinate_array):
    distance = np.sqrt((coordinate_array[:, 0] - latitude) ** 2 + (coordinate_array[:, 1] - longitude) ** 2)
    nearby = (distance > 0) & (distance <= 22.5)
    coherence.append(float((score_array[nearby] >= top_threshold).mean()) if nearby.any() else 0.0)
earth_df["spatial_coherence"] = coherence
earth_df["analog_score"] = (
    earth_df["marginal_similarity"] * (1 / 3)
    + earth_df["joint_similarity"] * (1 / 3)
    + earth_df["spatial_coherence"] * (1 / 3)
)
earth_df["rank"] = earth_df["analog_score"].rank(method="min", ascending=False).astype(int)
ranked_df = earth_df.sort_values(["analog_score", "joint_similarity"], ascending=False).reset_index(drop=True)
display(ranked_df.head(20))


## 3. CMR follow-up coverage near the best cells

This step uses the Earthdata Login secret only for NASA CMR inventory. It does not confuse “a MODIS granule
exists here” with a measured match. The actual numeric climate values above came from POWER.


In [ ]:
earthdata_token = None
try:
    from kaggle_secrets import UserSecretsClient
    earthdata_token = UserSecretsClient().get_secret("EARTHDATA_TOKEN")
except Exception:
    # Local execution can use an environment variable without printing it.
    earthdata_token = os.environ.get("EARTHDATA_TOKEN")

cmr_rows = []
cmr_headers = {"Authorization": f"Bearer {earthdata_token}"} if earthdata_token else {}
for row in ranked_df.head(20).itertuples(index=False):
    params = {
        "collection_concept_id": MODIS_ALBEDO_COLLECTION,
        "page_size": 10,
        "downloadable": "true",
        "bounding_box": f"{row.longitude - 7.5},{row.latitude - 7.5},{row.longitude + 7.5},{row.latitude + 7.5}",
    }
    try:
        response = requests.get(CMR_URL, params=params, headers=cmr_headers, timeout=45)
        response.raise_for_status()
        entries = response.json().get("feed", {}).get("entry", [])
        cmr_rows.append({
            "latitude": row.latitude, "longitude": row.longitude, "rank": row.rank,
            "cmr_status": "success", "granules_found": len(entries),
            "granule_ids": json.dumps([entry.get("id") for entry in entries]),
            "query_url": response.url,
        })
    except Exception as error:
        cmr_rows.append({
            "latitude": row.latitude, "longitude": row.longitude, "rank": row.rank,
            "cmr_status": "error", "granules_found": np.nan, "granule_ids": "[]", "query_url": repr(error),
        })
cmr_df = pd.DataFrame(cmr_rows)
if not earthdata_token:
    print("EARTHDATA_TOKEN was not available; CMR follow-up was skipped.")
display(cmr_df.head())


## Visual checks

The score map is a coarse 15° screening grid, not a final site boundary. The top rows need higher-resolution
Earth data and field/geological checks before being called analog sites.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(21, 6), constrained_layout=True)
scatter = axes[0].scatter(
    ranked_df["longitude"], ranked_df["latitude"], c=ranked_df["analog_score"],
    s=30, cmap="viridis", vmin=0, vmax=1,
)
axes[0].scatter(ranked_df.head(10)["longitude"], ranked_df.head(10)["latitude"], facecolors="none", edgecolors="red", s=90, linewidths=1.2)
axes[0].set_xlim(-180, 180); axes[0].set_ylim(-80, 80)
axes[0].set_title("Preliminary Earth analog score", loc="left", fontweight="bold")
axes[0].set_xlabel("Longitude"); axes[0].set_ylabel("Latitude")
fig.colorbar(scatter, ax=axes[0], label="score")

sns.histplot(data=ranked_df, x="analog_score", bins=20, color="#2563EB", ax=axes[1])
axes[1].axvline(ranked_df.head(10)["analog_score"].min(), color="#B91C1C", linestyle="--", label="top 10 boundary")
axes[1].set_title("Score distribution", loc="left", fontweight="bold")
axes[1].legend()

top_for_plot = ranked_df.head(20).copy()
sns.barplot(data=top_for_plot, y="rank", x="analog_score", color="#16A34A", ax=axes[2])
axes[2].set_title("Top ranked grid cells", loc="left", fontweight="bold")
axes[2].set_xlabel("analog score"); axes[2].set_ylabel("rank")
fig.savefig(FIGURE_DIR / "earth_dust_matching_overview.png", dpi=170, bbox_inches="tight")
plt.show()


In [ ]:
def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)): return [json_safe(v) for v in value]
    if isinstance(value, (np.integer,)): return int(value)
    if isinstance(value, (np.floating,)): return None if not np.isfinite(value) else float(value)
    if isinstance(value, (np.bool_,)): return bool(value)
    if pd.isna(value) if not isinstance(value, (str, bytes)) else False: return None
    return value


earth_df.to_parquet(OUTPUT_ROOT / "earth_power_climate_grid.parquet", index=False, compression="zstd")
ranked_df.to_parquet(OUTPUT_ROOT / "earth_dust_environment_matches.parquet", index=False, compression="zstd")
cmr_df.to_parquet(OUTPUT_ROOT / "earth_cmr_followup_inventory.parquet", index=False, compression="zstd")

summary = {
    "notebook": "08_earth_dust_environment_matching",
    "mars_source_regions": int(len(mars_supported_df)),
    "earth_power_cells": int(len(earth_df)),
    "earth_data_source": "NASA POWER climatology API",
    "cmr_followup_cells": int(len(cmr_df)),
    "cmr_successes": int(cmr_df["cmr_status"].eq("success").sum()) if len(cmr_df) else 0,
    "top_10": ranked_df.head(10)[["rank", "latitude", "longitude", "analog_score"]].to_dict(orient="records"),
    "interpretation": "preliminary environmental analog ranking; not proof of active aeolian transport",
    "proxy_limitations": proxy_mapping_df.to_dict(orient="records"),
}
(OUTPUT_ROOT / "matching_summary.json").write_text(json.dumps(json_safe(summary), indent=2), encoding="utf-8")
(OUTPUT_ROOT / "matching_summary.md").write_text(
    "# Earth dust-environment matching\n\n"
    f"Mars source regions: {summary['mars_source_regions']}\n\n"
    f"Earth POWER cells: {summary['earth_power_cells']}\n\n"
    "This is a preliminary environmental ranking. It does not prove active wind transport, dust mineralogy, or site suitability.\n",
    encoding="utf-8",
)
assert len(ranked_df) == len(earth_df)
assert ranked_df["rank"].is_unique
assert ranked_df["analog_score"].between(0, 1).all()
assert not (OUTPUT_ROOT / "matching_summary.json").read_text().find("your_token") >= 0
print(json.dumps({"top_10": summary["top_10"], "earth_cells": summary["earth_power_cells"]}, indent=2))
